# Outlier detection: three detectors that disagree

MichAl Academy, unit 2.24.

Outlier detection finds the rows least like the rest, without being told which
rows are unusual. That is the whole appeal and the whole problem: **least like
the rest** is not the same as **the thing you were looking for**, and nothing in
the method knows the difference.

Two questions here:

1. Do different detectors find the same rows?
2. If you do have labels, how much are they worth?


In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.covariance import EllipticEnvelope
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

warnings.filterwarnings("ignore")

digits = load_digits()
ordinary = digits.data[digits.target != 8]
print(f"{len(ordinary)} images, none of them an 8")
print("Nothing here is labelled unusual. That is the point.")


## Three detectors, one definition each

Each is asked for the most unusual 5% of the same rows. `contamination=0.05` is
not a discovery about the data, it is you telling the detector how many rows to
return, which lesson 2.16 would call a threshold.


In [ ]:
X = StandardScaler().fit_transform(ordinary)

flagged = {}
flagged["IsolationForest"] = IsolationForest(
    contamination=0.05, random_state=0).fit_predict(X) == -1
flagged["one-class SVM"] = OneClassSVM(nu=0.05, gamma="scale").fit_predict(X) == -1
flagged["local outlier factor"] = LocalOutlierFactor(
    contamination=0.05).fit_predict(X) == -1

for name, mask in flagged.items():
    print(f"{name:22s} flagged {mask.sum()} of {len(X)}")


They were each asked for the same number of rows. The question is whether they
picked the same ones.


In [ ]:
names = list(flagged)
print(f"{'':46s} both  either  overlap")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = flagged[names[i]], flagged[names[j]]
        both = int((a & b).sum())
        either = int((a | b).sum())
        print(f"{names[i]} and {names[j]:24s} {both:4d}  {either:6d}   {both/either:.3f}")

all_three = int(np.logical_and.reduce(list(flagged.values())).sum())
print(f"\nflagged by all three: {all_three}")


That overlap column is the finding. Each pair agrees on a small fraction of what
either one flagged, and only a handful of rows are picked by all three.

The detectors are not disagreeing about how many rows are unusual, because they
were told. They are disagreeing about **which**, because each is answering a
different question:

- **IsolationForest** asks how few random splits it takes to separate a row.
- **One-class SVM** asks whether a row falls outside a boundary drawn around the
  bulk of the data.
- **Local outlier factor** asks whether a row is in a thinner region than its own
  neighbours, which is a local question rather than a global one.

Picking a detector is picking a definition of unusual. It is not a tuning choice
and there is no default that is correct.


## What labels are worth

Now a case where labels do exist, so the two approaches can be compared on the
same data: breast cancer diagnoses, with the rare class thinned until it is
genuinely rare.

The comparison is what an analyst would actually do with either output.
**Investigate twenty cases.** How many real ones does each approach put in front
of you?


In [ ]:
data = load_breast_cancer()
y_all = (data.target == 0).astype(int)          # malignant is the rare class here

rng = np.random.default_rng(0)
common = np.flatnonzero(y_all == 0)
rare = rng.choice(np.flatnonzero(y_all == 1), 16, replace=False)
keep = np.concatenate([common, rare])

X_sub = StandardScaler().fit_transform(data.data[keep])
y_sub = y_all[keep]
print(f"{y_sub.sum()} real cases in {len(y_sub)} rows, "
      f"a base rate of {y_sub.mean():.4f}")


**Both approaches have to be scored on the same rows or the comparison means
nothing.** The supervised model needs rows it did not train on, so the data is
split in half and *everything* is judged on the held-back half. The unlabelled
detectors are fitted on the training half as well, even though they never look at
a label, so that neither side gets to see the test rows first.


In [ ]:
BUDGET = 20

X_tr, X_te, y_tr, y_te = train_test_split(
    X_sub, y_sub, test_size=0.5, stratify=y_sub, random_state=0)
print(f"{y_te.sum()} real cases in the {len(y_te)} held-back rows, "
      f"so {BUDGET} investigations is the budget for all of them\n")

unlabelled = {
    "IsolationForest": lambda: IsolationForest(random_state=0).fit(X_tr).score_samples(X_te),
    "one-class SVM": lambda: OneClassSVM(gamma="scale").fit(X_tr).score_samples(X_te),
    "local outlier factor": lambda: LocalOutlierFactor(novelty=True).fit(X_tr).score_samples(X_te),
    "elliptic envelope": lambda: EllipticEnvelope(support_fraction=0.9,
                                                  random_state=0).fit(X_tr).score_samples(X_te),
}

print("unlabelled, never shown a label")
best = 0
for name, fit_and_score in unlabelled.items():
    top = np.argsort(fit_and_score())[:BUDGET]     # lowest score is most unusual
    found = int(y_te[top].sum())
    best = max(best, found)
    print(f"  {name:22s} {found} of {BUDGET}")

forest = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_tr, y_tr)
top = np.argsort(-forest.predict_proba(X_te)[:, 1])[:BUDGET]
supervised = int(y_te[top].sum())

print(f"\nsupervised forest, trained on labels   {supervised} of {BUDGET}")
print(f"best unlabelled                        {best} of {BUDGET}")
print(f"what the labels bought                 {supervised - best}")


## What to take away

- Different detectors flag different rows and each is answering its own question
  about what unusual means. Run more than one and look at the overlap before
  trusting any of them.
- `contamination` is a threshold. It decides how many rows come back, and it is
  your decision rather than the data's.
- Labels are worth having, and on a problem this rare they are worth less than
  people expect. An unlabelled detector is not a poor substitute for a model; it
  is what you use before anybody has labelled anything, and it is how the first
  labels get made.

The honest use of these methods is to produce a queue for a person to work
through, which is lesson 2.12's precision at a fixed budget, not a verdict.
